In [ ]:
import scipy.io as scio
import pandas as pd
import os
import numpy as np
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import interpolate
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torch.utils.data as Data
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from torch import einsum
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold, LeaveOneGroupOut
import copy
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, accuracy_score
#from sklearn import preprocessing
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from tqdm import tqdm, trange
from torcheeg.datasets.constants.emotion_recognition import format_region_channel_list
# from ConLoss import SupConLoss
import random
# import Module as md
import mne

from torcheeg.datasets.constants.emotion_recognition.deap import DEAP_GENERAL_REGION_LIST
from torcheeg.models import (EEGNet, FBCNet, TSCeption)
from torcheeg.models import DGCNN, ArjunViT, STNet, FBCNet, GRU, LSTM
# import MI as mi

G:\python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data=torch.from_numpy(np.load(f'data/train/eeg_data.npy'))
train_dec_label=torch.from_numpy(np.load(f'data/train/dec_label.npy'))
train_emo_label=torch.from_numpy(np.load(f'data/train/emo_label.npy'))
test_data=torch.from_numpy(np.load(f'data/test/eeg_data.npy'))
test_dec_label=torch.from_numpy(np.load(f'data/test/dec_label.npy'))
test_emo_label=torch.from_numpy(np.load(f'data/test/emo_label.npy'))
train_data.shape,train_dec_label.shape,train_emo_label.shape,test_data.shape,test_dec_label.shape,test_emo_label.shape

(torch.Size([1698, 1, 62, 875]),
 torch.Size([1698]),
 torch.Size([1698]),
 torch.Size([425, 1, 62, 875]),
 torch.Size([425]),
 torch.Size([425]))

In [18]:
'''
dgcnn数据设置
'''
train_data_dgcnn = torch.squeeze(train_data, dim=1).float()
test_data_dgcnn=torch.squeeze(test_data,dim=1).float()


In [34]:
'''
FBCNet数据准备,9个波段
'''
from scipy.signal import butter, lfilter
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y
train_data_freq = np.empty((len(train_data), 9, 62, 875),dtype=np.float32)
test_data_freq = np.empty((len(test_data), 9, 62, 875),dtype=np.float32)
for trial in range(train_data.shape[0]):
    for channels in range(train_data.shape[2]):
        data_1 = butter_bandpass_filter(train_data[trial, :, channels], 4, 8, 250, order=3)
        data_2 = butter_bandpass_filter(train_data[trial, :, channels], 8, 12, 250, order=3)
        data_3 = butter_bandpass_filter(train_data[trial, :, channels], 12, 16, 250, order=3)
        data_4 = butter_bandpass_filter(train_data[trial, :, channels], 16, 20, 250, order=3)
        data_5 = butter_bandpass_filter(train_data[trial, :, channels], 20, 24, 250, order=3)
        data_6 = butter_bandpass_filter(train_data[trial, :, channels], 24, 28, 250, order=3)
        data_7 = butter_bandpass_filter(train_data[trial, :, channels], 28, 32, 250, order=3)
        data_8 = butter_bandpass_filter(train_data[trial, :, channels], 32, 36, 250, order=3)
        data_9 = butter_bandpass_filter(train_data[trial, :, channels], 36, 40, 250, order=3)
        
        train_data_freq[trial,0, channels] = data_1
        train_data_freq[trial,1, channels] = data_2
        train_data_freq[trial,2, channels] = data_3
        train_data_freq[trial,3, channels] = data_4
        train_data_freq[trial,4, channels] = data_5
        train_data_freq[trial,5, channels] = data_6
        train_data_freq[trial,6, channels] = data_7
        train_data_freq[trial,7, channels] = data_8
        train_data_freq[trial,8, channels] = data_9
        
for trial in range(test_data.shape[0]):
    for channels in range(test_data.shape[2]):
        data_1 = butter_bandpass_filter(test_data[trial, :, channels], 4, 8, 250, order=3)
        data_2 = butter_bandpass_filter(test_data[trial, :, channels], 8, 12, 250, order=3)
        data_3 = butter_bandpass_filter(test_data[trial, :, channels], 12, 16, 250, order=3)
        data_4 = butter_bandpass_filter(test_data[trial, :, channels], 16, 20, 250, order=3)
        data_5 = butter_bandpass_filter(test_data[trial, :, channels], 20, 24, 250, order=3)
        data_6 = butter_bandpass_filter(test_data[trial, :, channels], 24, 28, 250, order=3)
        data_7 = butter_bandpass_filter(test_data[trial, :, channels], 28, 32, 250, order=3)
        data_8 = butter_bandpass_filter(test_data[trial, :, channels], 32, 36, 250, order=3)
        data_9 = butter_bandpass_filter(test_data[trial, :, channels], 36, 40, 250, order=3)
        
        test_data_freq[trial,0, channels] = data_1
        test_data_freq[trial,1, channels] = data_2
        test_data_freq[trial,2, channels] = data_3
        test_data_freq[trial,3, channels] = data_4
        test_data_freq[trial,4, channels] = data_5
        test_data_freq[trial,5, channels] = data_6
        test_data_freq[trial,6, channels] = data_7
        test_data_freq[trial,7, channels] = data_8
        test_data_freq[trial,8, channels] = data_9

        
train_data_freq = torch.from_numpy(train_data_freq).float()
test_data_freq=torch.from_numpy(test_data_freq).float()

In [35]:
np.save('data_training/FBCNet_features_training.npy',train_data_freq)
np.save('data_training/FBCNet_features_testing.npy',test_data_freq)

In [22]:
train_data_freq=torch.from_numpy(np.load('data_training/FBCNet_features_training.npy'))
test_data_freq=torch.from_numpy(np.load('data_training/FBCNet_features_testing.npy'))

In [7]:
eeg_data_now = mne.io.read_raw_eeglab('F:/2.科研/2023-10-24 数据处理/2023-11-29 脑电数据处理/3.脑电预处理set文件/preprocessed_01102201.set', preload=True)
# 要删除的通道名字列表
channels_to_remove = ['M1', 'M2','HEO','VEO','Trigger']

# 使用 pick_channels 删除指定的通道
eeg_data_now.pick_channels(ch_names=[ch for ch in eeg_data_now.ch_names if ch not in channels_to_remove])

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


<RawEEGLAB | preprocessed_01102201.set, 62 x 164825 (659.3 s), ~78.1 MB, data loaded>

In [ ]:
criterion1 = nn.CrossEntropyLoss()

learning_rate = 0.0001
epoch_size = [40,40,40,40,40,40]
# epoch_size = 11
device = torch.device("cuda:0")
# kf = KFold(n_splits=5, shuffle=True, random_state=123)
# logo = LeaveOneGroupOut()


In [ ]:

model_name_list = ['ArjunViT','LSTM','GRU', 'DGCNN', 'FBCNet',"EEGNet"]

save = True


In [ ]:


for m in range(len(model_name_list)):
    model_name=model_name_list[m]
    acc_all = []
    f1_all = []
    recall_all = []
    precision_all = []
    for k in range(5):
    
        """ Build Network """
        if model_name == "EEGNet": 
            model = EEGNet(chunk_size=875,
                       num_electrodes=62,
                       dropout=0.2,
                       kernel_1=64,
                       kernel_2=32,
                       F1=8,
                       D=2,
                       F2=16,
                       num_classes=2).to(device)

        if model_name == "DGCNN":    
            model = DGCNN(in_channels=875,
                      num_electrodes=62,
                      hid_channels=64,
                      num_layers=4,
                      num_classes=2).to(device)

    #         eeg_data_dgcnn = torch.squeeze(eeg_data, dim=1)
        if model_name == 'ArjunViT':
            model = ArjunViT(chunk_size=875,
                             t_patch_size=25,
                             num_electrodes=62,
                             dropout=0.25,
                             embed_dropout=0.25,
                             num_classes=2).to(device)

        if model_name == 'FBCNet':
            model = FBCNet(num_classes=2,
                           num_electrodes=62,
                           stride_factor=5,
                           chunk_size=875,
                           in_channels=9,
                           num_S=32).to(device)
        if model_name == 'GRU':    

            model = GRU(num_electrodes=62, hid_channels=32, num_classes=2).to(device)
        if model_name == 'LSTM':

            model = LSTM(num_electrodes=62, hid_channels=32, num_classes=2).to(device)
        
        """ Optimizer """
        optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate#, weight_decay=0.0005
                                    )
        #训练集
        if model_name == 'FBCNet':
            train_set = TensorDataset(train_data_freq, train_dec_label)
            train_loader = Data.DataLoader(train_set, batch_size=32)

        if model_name == "DGCNN"or model_name == 'ArjunViT' or model_name == 'LSTM' or model_name == 'GRU': 
            train_set=TensorDataset(train_data_dgcnn, train_dec_label)
            train_loader = Data.DataLoader(train_set, batch_size=32)

        if model_name == 'EEGNet' :
            train_set = TensorDataset(train_data, train_dec_label)
            train_loader = Data.DataLoader(train_set, batch_size=32)
            
        #测试集    
        if model_name == 'DGCNN' or   model_name == 'ArjunViT' or   model_name == 'LSTM' or   model_name == 'GRU':
            test_set = TensorDataset(test_data_dgcnn, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)            
        if model_name == 'FBCNet':
            test_set = TensorDataset(test_data_freq, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)

        if model_name == 'EEGNet' or model_name == 'MISICR' :
            test_set = TensorDataset(test_data, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)

        for i in range(epoch_size[m]):                                          #多个epoch
            loop = tqdm(enumerate(train_loader), total=len(train_loader))
            model.train()
            learning_rate=0.99*learning_rate

            train_loss = 0.0
            train_acc_task = 0.0
            for step, (x, y1) in loop:
                x, y1 =  Variable(x).to(device), Variable(y1).to(device)
                optimizer.zero_grad()

                if model_name == 'EEGNet' or model_name =="FBCNet" or model_name == 'DGCNN' or model_name == 'ArjunViT' or model_name == 'LSTM' or model_name == 'GRU': 

                    pred_task = model(x)
                    loss = criterion1(pred_task, y1.long())

                train_loss += loss.item()
                

                pred_task = torch.max(pred_task, 1)[1]
                train_correct_task = (pred_task == y1).sum()

                train_acc_task += train_correct_task.item()

                loss.backward()
                optimizer.step()
                loop.set_description(f'Epoch [{i+1} / {epoch_size[m]}]')
                loop.set_postfix({
                        'loss' : '{:.6f}'.format(train_loss/len(train_set)),
                        'acc_task' : '{:.6f}'.format(train_acc_task*100/len(train_set))
                                                    })
                if i+1 == epoch_size[m] and save == True:   
                    model_path = './model_parameter/%s_%d.pkl' % (model_name,k+1)  #文件夹名称
                    os.makedirs('./model_parameter', exist_ok=True)   #创建文件夹
                    state = {'model':model.state_dict()
                            }
                    torch.save(state,model_path)
                    
                    
        prob_all = []
        label_all = []
        with torch.no_grad():
            model.eval()  
            for x, y1 in test_loader:
                x, y1 = Variable(x).to(device), Variable(y1).to(device)
                pred_task = model(x)
                prob = pred_task.cpu().numpy()
                prob_all.extend(np.argmax(prob,axis=1)) #求每一行的最大值索引
                label_all.extend(y1.cpu().numpy())
            acc_now = accuracy_score(y_true=prob_all, y_pred=label_all)
            f1_now = f1_score(y_true=prob_all, y_pred=label_all, average="macro")
            recall_now = recall_score(y_true=prob_all, y_pred=label_all, average="macro")
            precision_now = precision_score(y_true=prob_all, y_pred=label_all, average="macro")
        acc_all.append(acc_now)
        f1_all.append(f1_now)
        recall_all.append(recall_now)
        precision_all.append(precision_now)
        print('accuracy:', np.around(acc_now*100, 2))
        print('precision:', np.around(precision_now*100, 2))
        print('recall:', np.around(recall_now*100, 2))
        print('f1:', np.around(f1_now*100, 2))
        

        
    print('-'*10, model_name, '-'*10)
    print('accuracy:', np.around(np.mean(acc_all)*100, 2), np.std(acc_all))
    print('precision:', np.around(np.mean(precision_all)*100, 2), np.std(precision_all))
    print('recall:', np.around(np.mean(recall_all)*100, 2), np.std(recall_all))
    print('f1:', np.around(np.mean(f1_all)*100, 2), np.std(f1_all))
    
    
    

In [ ]:
#呈现模型结果
import joblib

for m in range(len(model_name_list)):
    model_name=model_name_list[m]
    acc_all = []
    f1_all = []
    recall_all = []
    precision_all = []
    
    
    for k in range(5):
        load_path = './model_parameter/%s_%d.pkl' % (model_name,k+1)
        state_dict = torch.load(load_path)
        """ Build Network """
        if model_name == "EEGNet": 
            model = EEGNet(chunk_size=875,
                       num_electrodes=62,
                       dropout=0.2,
                       kernel_1=64,
                       kernel_2=32,
                       F1=8,
                       D=2,
                       F2=16,
                       num_classes=2).to(device)

        if model_name == "DGCNN":    
            model = DGCNN(in_channels=875,
                      num_electrodes=62,
                      hid_channels=64,
                      num_layers=4,
                      num_classes=2).to(device)

    #         eeg_data_dgcnn = torch.squeeze(eeg_data, dim=1)
        if model_name == 'ArjunViT':
            model = ArjunViT(chunk_size=875,
                             t_patch_size=25,
                             num_electrodes=62,
                             dropout=0.25,
                             embed_dropout=0.25,
                             num_classes=2).to(device)

        if model_name == 'FBCNet':
            model = FBCNet(num_classes=2,
                           num_electrodes=62,
                           stride_factor=5,
                           chunk_size=875,
                           in_channels=9,
                           num_S=32).to(device)
        if model_name == 'GRU':    

            model = GRU(num_electrodes=62, hid_channels=32, num_classes=2).to(device)
        if model_name == 'LSTM':

            model = LSTM(num_electrodes=62, hid_channels=32, num_classes=2).to(device)
        
        """ Optimizer """
        optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate#, weight_decay=0.0005
                                    )

            
        #测试集    
        if model_name == 'DGCNN' or   model_name == 'ArjunViT' or   model_name == 'LSTM' or   model_name == 'GRU':
            test_set = TensorDataset(test_data_dgcnn, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)            
        if model_name == 'FBCNet':
            test_set = TensorDataset(test_data_freq, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)

        if model_name == 'EEGNet' or model_name == 'MISICR' :
            test_set = TensorDataset(test_data, test_dec_label)
            test_loader = Data.DataLoader(test_set, batch_size=32)

        model.load_state_dict(state_dict['model'])
        
        
                    
        prob_all = []
        label_all = []
        with torch.no_grad():
            model.eval()  
            for x, y1 in test_loader:
                x, y1 = Variable(x).to(device), Variable(y1).to(device)
                pred_task = model(x)
                prob = pred_task.cpu().numpy()
                prob_all.extend(np.argmax(prob,axis=1)) #求每一行的最大值索引
                label_all.extend(y1.cpu().numpy())
            acc_now = accuracy_score(y_true=prob_all, y_pred=label_all)
            f1_now = f1_score(y_true=prob_all, y_pred=label_all, average="macro")
            recall_now = recall_score(y_true=prob_all, y_pred=label_all, average="macro")
            precision_now = precision_score(y_true=prob_all, y_pred=label_all, average="macro")
        acc_all.append(acc_now)
        f1_all.append(f1_now)
        recall_all.append(recall_now)
        precision_all.append(precision_now)
        print('accuracy:', np.around(acc_now*100, 2))
        print('precision:', np.around(precision_now*100, 2))
        print('recall:', np.around(recall_now*100, 2))
        print('f1:', np.around(f1_now*100, 2))
        

        
    print('-'*10, model_name, '-'*10)
    print('accuracy:', np.around(np.mean(acc_all)*100, 2), np.std(acc_all))
    print('precision:', np.around(np.mean(precision_all)*100, 2), np.std(precision_all))
    print('recall:', np.around(np.mean(recall_all)*100, 2), np.std(recall_all))
    print('f1:', np.around(np.mean(f1_all)*100, 2), np.std(f1_all))
    
    
    